In [ ]:


# Download the dataset, setup packages
import os
import cv2
import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score

if not os.path.exists('dataset.zip'):
  !gdown 1_pRKXtYRjWjY0seYqyx25nOxjtr-mHYg
  !unzip -q -u dataset.zip
else:
  print('Already downloaded')

Already downloaded


In [ ]:
# Some helper functions for your project
def load_dataset(class_name = 'pasta'):
  assert class_name in ['pasta', 'screws']
  dir = './dataset/'+class_name+'/'
  training_images = []
  testing_images = []
  testing_labels = []
  for file_name in os.listdir(dir+'train/good/'):
    training_images.append(cv2.cvtColor(cv2.imread(dir+'train/good/'+file_name), cv2.COLOR_BGR2RGB))
  for file_name in os.listdir(dir+'test/good/'):
    testing_images.append(cv2.cvtColor(cv2.imread(dir+'test/good/'+file_name), cv2.COLOR_BGR2RGB))
    testing_labels.append(0)
  for file_name in os.listdir(dir+'test/bad/'):
    testing_images.append(cv2.cvtColor(cv2.imread(dir+'test/bad/'+file_name), cv2.COLOR_BGR2RGB))
    testing_labels.append(1)
  return np.array(training_images)/255., np.array(testing_images)/255., np.array(testing_labels)

def basic_evaluation(predictions : np.ndarray, targets : np.ndarray):
  print(targets)
  print(predictions)
  print('AUROC Score:', roc_auc_score(targets, predictions))

  # threshold = np.percentile(predictions, 45)
  # binary_predictions = (predictions > threshold).astype(int)

  # accuracy = accuracy_score(targets, binary_predictions)
  # f1 = f1_score(targets, binary_predictions)
  # precision = precision_score(targets, binary_predictions)
  # recall = recall_score(targets, binary_predictions)

  # print(f'{threshold}')
  # print(f"Accuracy: {accuracy:.4f}")
  # print(f"F1 Score: {f1:.4f}")
  # print(f"Precision: {precision:.4f}")
  # print(f"Recall: {recall:.4f}")

  # goes over entire AURUC curve (in 0.01 increments) and calculates Accuracy, F1 score, Precision, and Recall to find optimal results
  for i in range(0, 100):
    threshold = np.percentile(predictions, i)
    binary_predictions = (predictions > threshold).astype(int)

    accuracy = accuracy_score(targets, binary_predictions)
    f1 = f1_score(targets, binary_predictions)
    precision = precision_score(targets, binary_predictions)
    recall = recall_score(targets, binary_predictions)

    print(f'Threshold: {i}%')
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")


In [ ]:
from sklearn.svm import OneClassSVM
import torch
from torchvision import models
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [ ]:


def save_image(image, title="Image", filename='output.png'):
  if isinstance(image, torch.Tensor):
    image = image.permute(1,2,0).detach().numpy()
  plt.imshow(image)
  plt.title(title)
  plt.axis('off')
  plt.savefig(filename, bbox_inches='tight', pad_inches=0)
  plt.show()

class ConvAutoencoder(nn.Module):
  def __init__(self):
    super(ConvAutoencoder, self).__init__()

    kernel_size = 3
    stride = 2
    padding = 1
    latent_dim = 256
    # ENCODER: Compresses the input image
    self.encoder = nn.Sequential(
        nn.Conv2d(in_channels=3, out_channels=16, kernel_size=kernel_size, stride=stride, padding=padding),
        nn.ReLU(),
        nn.Conv2d(in_channels=16, out_channels=32, kernel_size=kernel_size, stride=stride, padding=padding),
        nn.ReLU(),
        nn.Conv2d(in_channels=32, out_channels=latent_dim, kernel_size=kernel_size, stride=stride, padding=padding),
        nn.ReLU(),


    )

    # DECODER: Reconstruct the image back to original shape
    self.decoder = nn.Sequential(
        nn.ConvTranspose2d(in_channels=latent_dim, out_channels=32, kernel_size=kernel_size, stride=stride, padding=padding, output_padding=1),
        nn.ReLU(),
        nn.ConvTranspose2d(in_channels=32, out_channels=16, kernel_size=kernel_size, stride=stride, padding=padding, output_padding=1),
        nn.ReLU(),
        nn.ConvTranspose2d(in_channels=16, out_channels=3, kernel_size=kernel_size, stride=stride, padding=padding, output_padding=1),

        nn.Sigmoid(),
    )

  def forward(self, x):
    latent_rep = self.encoder(x)
    reconstructed = self.decoder(latent_rep)

    return reconstructed, latent_rep


class AnomalyDetector:
  # Creates new instance of ConvAutoencoder model.
  # Moves the model to the CPU for training
  def __init__(self,):
    self.model = ConvAutoencoder()
    self.device = torch.device("cpu")
    self.model.to(self.device)

  def save_latent_representation(self, latent_rep):
    latent_image = latent_rep[0, 0].detach().numpy()
    plt.imshow(latent_image, cmap='viridis')
    plt.title("First Channel (0)")
    plt.axis('off')
    plt.savefig('latent_representation_first.png', bbox_inches='tight', pad_inches=0)
    plt.show()

    # Last channel
    last_ch = latent_rep.shape[1] - 1
    latent_image_last = latent_rep[0, last_ch].detach().numpy()
    plt.imshow(latent_image_last, cmap='viridis')
    plt.title(f"Last Channel ({last_ch})")
    plt.axis('off')
    plt.savefig('latent_representation_last.png', bbox_inches='tight', pad_inches=0)
    plt.show()


  def preprocess(self, images):
    proccessed_images = []
    for idx, image in enumerate(images):
      image = cv2.resize(image, (128, 128))
      image = torch.tensor(image.transpose(2,0,1), dtype=torch.float32)
      #image = image / 255.0
      proccessed_images.append(image)

    return torch.stack(proccessed_images)

  def create_model(self, train_images):
    # Preprocesses trainign images + wraps them in PoyTorch Dataset and Dataloader
    train_tensor = self.preprocess(train_images)
    dataset = TensorDataset(train_tensor)
    loader = DataLoader(dataset, batch_size=8, shuffle=True)

    # Uses mean squared error as the loss function
    criterion = nn.MSELoss()

    # Uses adam optimizer (first-order gradient-based optimizer)
    lr=0.001
    optimizer = optim.Adam(self.model.parameters(), lr=lr)

    # puts model into training mode
    self.model.train()

    n_epoch = 1000

    for epoch in range(n_epoch):
        total_loss = 0
        for data in loader:                       # added
            inputs = data[0].to(self.device)
            optimizer.zero_grad()
            outputs, _ = self.model(inputs)
            loss = criterion(outputs, inputs)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

  def predict(self, test_images, testing_labels):
    test_tensor = self.preprocess(test_images).to(self.device)
    self.model.eval()
    with torch.no_grad():
      recon, latent_rep = self.model(test_tensor)
      #self.save_latent_representation(latent_rep)

      #save_image(recon[0], title="Reconstructed Image", filename="reconstructed_image.png")

    # calculate metrics

    mse = torch.mean((recon - test_tensor) ** 2, dim=[1,2,3]).cpu().numpy()
    return mse




In [ ]:


def do_analysis(ad, class_name):
  training_images, testing_images, testing_labels = load_dataset(class_name=class_name)
  ad.create_model(training_images)
  predictions = ad.predict(testing_images, testing_labels)
  basic_evaluation(predictions, testing_labels)
do_analysis(AnomalyDetector(), 'screws')
do_analysis(AnomalyDetector(), 'pasta')


Epoch 1, Loss: 0.1911
Epoch 2, Loss: 0.1856
Epoch 3, Loss: 0.1873
Epoch 4, Loss: 0.1749
Epoch 5, Loss: 0.1646
Epoch 6, Loss: 0.1590
Epoch 7, Loss: 0.1511
Epoch 8, Loss: 0.1398
Epoch 9, Loss: 0.1334
Epoch 10, Loss: 0.1200
Epoch 11, Loss: 0.1083
Epoch 12, Loss: 0.0937
Epoch 13, Loss: 0.0776
Epoch 14, Loss: 0.0644
Epoch 15, Loss: 0.0592
Epoch 16, Loss: 0.0567
Epoch 17, Loss: 0.0543
Epoch 18, Loss: 0.0512
Epoch 19, Loss: 0.0487
Epoch 20, Loss: 0.0473
Epoch 21, Loss: 0.0447
Epoch 22, Loss: 0.0431
Epoch 23, Loss: 0.0418
Epoch 24, Loss: 0.0391
Epoch 25, Loss: 0.0381
Epoch 26, Loss: 0.0357
Epoch 27, Loss: 0.0344
Epoch 28, Loss: 0.0323
Epoch 29, Loss: 0.0312
Epoch 30, Loss: 0.0296
Epoch 31, Loss: 0.0284
Epoch 32, Loss: 0.0275
Epoch 33, Loss: 0.0267
Epoch 34, Loss: 0.0259
Epoch 35, Loss: 0.0245
Epoch 36, Loss: 0.0236
Epoch 37, Loss: 0.0229
Epoch 38, Loss: 0.0222
Epoch 39, Loss: 0.0216
Epoch 40, Loss: 0.0210
Epoch 41, Loss: 0.0193
Epoch 42, Loss: 0.0192
Epoch 43, Loss: 0.0188
Epoch 44, Loss: 0.01